In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Generación de los datos sintéticos
from data_generation.data_config import DATA_CONFIG
from data_generation.PanelCreditSimulator import PanelCreditSimulator
from playground.generate_simple_panel import generate_panel

# Divisón de IDs en train y test
from splits.split_generator import SplitGenerator

In [ ]:
# Paso 1: generar los datos (por ahora hacemos uno muy sintético)
panel = generate_panel()

# Caso real
# panel_simulator = PanelCreditSimulator(DATA_CONFIG)
# panel = panel_simulator.simulate_panel()

In [ ]:
# Paso 1.1: análisis exploratorio de los datos
print(panel.head())
print()
print(panel.info())
print()
print(list(panel.columns))

In [ ]:
# Paso 2: generar el split train/test. El split_generator ya está configurado
# para que al train vayan todos los tratados y un porcentaje de los nini, y al
# test vayan todos los controles y el resto de los nini
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=13)
train_ids, test_ids = split_generator.generate()

In [ ]:
last = panel.sort_values('t').groupby('firm_id').last()
status = last[['treated', 'control', 'cohort']].reset_index()
status

In [ ]:
# Paso 2.1: revisar que el split se hizo correctamente
split = split_generator.split

train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

for id in treated:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == True, f"Firm {id} is not treated"
    assert firm["control"].iloc[0] == False, f"Firm {id} is control"

for id in control:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["control"].iloc[0] == True, f"Firm {id} is not control"
    assert firm["treated"].iloc[0] == False, f"Firm {id} is not treated"

for id in nini_train + nini_test:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == False and firm["control"].iloc[0] == False, f"Firm {id} is not NiNi"